## Módulo 3 - Review Embeddings

### Introducción

En esta sección construimos las representaciones semánticas (embeddings) que van a permitir la búsqueda por similitud conceptual del sistema. La idea central es transformar el texto de las reviews en vectores numéricos que capturen su significado, de modo que una consulta en lenguaje natural como "hotel tranquilo con linda vista" pueda compararse contra las reviews de cada hotel y recuperar los más afines, aunque no compartan exactamente las mismas palabras.

Para generar estas representaciones utilizamos un modelo de tipo sentence-transformer: un modelo encoder only pre-entrenado (arquitectura tipo BERT) que recibe un texto y devuelve un vector de dimensión fija que resume su significado. A diferencia de un LLM generativo, este modelo no produce texto ni razona: solo transforma texto en vectores. Esto lo hace órdenes de magnitud más liviano, gratuito y ejecutable localmente sobre la totalidad del dataset sin restricciones de costo.

Adoptamos un enfoque pragmático: generamos los embeddings directamente sobre el texto positivo de las reviews y evaluamos su calidad antes de incorporar etapas adicionales de procesamiento. Esta primera iteración nos permite validar el enfoque más simple y económico antes de complejizar el pipeline.

Para esta primera iteración decidimos generar los embeddings únicamente a partir del texto positivo de las reviews. La razón es que las consultas de los usuarios a un sistema de recomendación se formulan siempre en términos de aquello que desean (por ejemplo, "hotel con buen desayuno y pileta"), y nunca en términos de lo que quieren evitar. El texto positivo concentra, por lo tanto, la señal directamente comparable contra ese tipo de consultas.

Dejamos constancia de que esta es una simplificación deliberada de la primera iteración. El texto negativo contiene información valiosa para penalizar hoteles (un hotel con muchas quejas sobre el wifi debería bajar en una búsqueda de "buen wifi", aunque nadie lo haya elogiado explícitamente). Su incorporación queda planteada como una mejora natural del sistema en caso de que la evaluación de esta primera versión lo justifique.

### Set-Up Inicial

In [1]:
# Importamos librerías
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [2]:
# Cargamos el dataset final del EDA desde una ruta relativa al repo.
# El notebook está en Scripts/, por eso usamos Path.cwd().parent para subir a la raíz del proyecto.

ruta_dataset_final = Path.cwd().parent / "Data" / "Final" / "eda_final_dataset.parquet"

df = pd.read_parquet(ruta_dataset_final)

### Preparación Del Texto A Vectorizar

In [3]:
# Head
df.head()

,hotel_id_review,fecha_review,fecha_checkout,idioma,overall_rating,score_ubicacion,score_limpieza,score_habitacion,score_servicio,score_personal,...,name,city,country,star_rating,hotel_type,amenities,meal_plans,len_positivo,len_negativo,destino
0,4669631,"10 febrero, 2025","31 enero, 2025",ES,95.56,100,100,100,100,100,...,Buona Vitta Gramado Resort & Spa by Gramado Parks,Gramado,Brasil,5.0,Hoteles,"{'Gimnasio','Valet parking','Servicio de spa',...",{'CONTINENTAL_BREAKFAST'},54,0,Brasil - Gramado
1,231021,"10 febrero, 2025","6 febrero, 2025",ES,100.00,100,100,100,100,100,...,Majestic Rio Palace Hotel,Rio De Janeiro,Brasil,3.0,Hoteles,{'Baño adaptado para personas con movilidad re...,{'BUFFET_BREAKFAST'},330,0,Brasil - Rio De Janeiro
2,265830,"10 febrero, 2025","4 febrero, 2025",ES,100.00,100,100,100,100,100,...,Melia Casa Maya Cancun All Inclusive,Cancún,México,4.0,Hoteles,"{'Terraza','Periodo de desinfección entre esta...",{'ALL_INCLUSIVE'},49,0,México - Cancún
3,358839,"10 febrero, 2025","9 febrero, 2025",ES,91.11,80,100,100,100,100,...,Gran Hotel Continental,Mar Del Plata,Argentina,3.0,Hoteles,"{'Sala de reuniones','Bar','Servicio de conser...",{'BUFFET_BREAKFAST'},71,49,Argentina - Mar Del Plata
4,999361,"10 febrero, 2025","10 febrero, 2025",ES,82.22,100,100,60,100,100,...,Ibis Budget Rio de Janeiro - Praia de Botafogo,Rio De Janeiro,Brasil,2.0,Hoteles,"{'Recepción 24 hrs','Seguridad 24 hrs','Restau...","{'BREAKFAST','BUFFET_BREAKFAST','ROOM_ONLY'}",12,7,Brasil - Rio De Janeiro


In [4]:
# En el EDA habíamos filtrado filas que tengan ambos textos (positivo y negativo) nulos
# Ahora filtramos filas con texto positivo nulo pues no tiene sentido procesarlo
mask = df['texto_positivo'].notna() # Máscara booleana
df = df.loc[mask, :].reset_index(drop = True)
df.shape

(251357, 37)

In [5]:
# Sacamos espacios sobrantes (inicio/fin, saltos de línea) para normalizar el texto antes de vectorizar
df['texto_positivo'] = df['texto_positivo'].str.strip()

In [6]:
# Descartamos las reviews que quedaron como string vacío tras la limpieza
df = df[df['texto_positivo'] != ''].reset_index(drop = True)

In [7]:
# Imprimimos
print(f'Reviews a vectorizar: {df.shape[0]:,}')
print(f'Hoteles únicos: {df["hotel_id_review"].nunique():,}')
df[['hotel_id_review', 'texto_positivo']].head()

Reviews a vectorizar: 251,332
Hoteles únicos: 2,523


,hotel_id_review,texto_positivo
0,4669631,"El gran espacio que dispone, el restaurante mu..."
1,231021,"La atención del personal es espectacular, son ..."
2,265830,"Ubicación,la playa y las instalaciones en general"
3,358839,Excelente el personal. La habitación triple su...
4,999361,Localização.


### Elección Del Modelo De Embeddings E Inferencia

Para vectorizar las reviews necesitamos un modelo de sentence embeddings que cumpla tres requisitos:

1. Multilingüe: nuestro dataset contiene reviews en español y portugués. El modelo debe ubicar en posiciones cercanas del espacio vectorial a frases equivalentes aunque estén en distinto idioma (por ejemplo, "buen desayuno" y "bom café da manhã" deberían quedar próximas).

2. Buen balance calidad / velocidad: vamos a vectorizar ~250.000 reviews, por lo que un modelo demasiado pesado haría el proceso impracticable en Colab.

3. Pre-entrenado y de uso libre: no entrenamos el modelo; lo usamos tal cual fue publicado por la comunidad.

El modelo elegido es paraphrase-multilingual-MiniLM-L12-v2, de la familia sentence-transformers. Es un encoder de ~118M de parámetros, entrenado sobre más de 50 idiomas (incluidos español y portugués) mediante un objetivo de aprendizaje contrastivo que optimiza directamente la similitud semántica entre oraciones. Genera vectores de 384 dimensiones, un tamaño suficientemente expresivo para capturar el significado de una review y, a la vez, lo bastante compacto como para que la búsqueda por similitud sea rápida.

In [8]:
# Instalamos la librería (solo la primera vez)
%pip install sentence-transformers -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
# Importamos librería
from sentence_transformers import SentenceTransformer

# Cargamos el modelo multilingüe pre-entrenado
# La primera vez lo descarga (~470 MB); después queda cacheado en la sesión
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print('Modelo cargado correctamente.')
print(f'Dimensión de los embeddings: {modelo.get_sentence_embedding_dimension()}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo cargado correctamente.
Dimensión de los embeddings: 384


/var/folders/3_/n3rxgyx52s38d1zrs4t2d3v00000gn/T/ipykernel_16989/3385902210.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Dimensión de los embeddings: {modelo.get_sentence_embedding_dimension()}')


Antes de vectorizar las ~250.000 reviews, vamos a hacer un test rápido para confirmar que el modelo entiende bien español y portugués y que la similitud semántica entre ambos idiomas funciona.

In [10]:
# Importamos librería
from sentence_transformers import util

# Frases de prueba: mismas ideas en ES y PT, más una distinta
frases = [
    "El desayuno estaba riquísimo", # ES
    "O café da manhã estava delicioso", # PT (misma idea)
    "La habitación tenía vista al mar", # ES (idea distinta)
]

emb = modelo.encode(frases)

# Similitud coseno entre todas las combinaciones
sim = util.cos_sim(emb, emb)

print("Matriz de similitud:")
print(np.round(sim.numpy(), 3))

Matriz de similitud:
[[1.    0.985 0.176]
 [0.985 1.    0.187]
 [0.176 0.187 1.   ]]


La prueba confirma que el modelo captura correctamente la equivalencia semántica entre idiomas: frases con el mismo significado en español y portugués obtienen una similitud de 0.98, mientras que frases con ideas distintas se mantienen por debajo de 0.22. Esto valida el uso del modelo para la búsqueda semántica sobre nuestro dataset bilingüe.

In [11]:
# Convertimos la columna de texto a una lista (formato esperado por el modelo)
textos = df['texto_positivo'].tolist()

print(f'Total de reviews a vectorizar: {len(textos):,}')

Total de reviews a vectorizar: 251,332


Generamos los embeddings de todas las reviews

* batch_size: cuántas reviews procesa la GPU en paralelo por lote

* show_progress_bar: muestra una barra para ver el avance

* convert_to_numpy: devuelve el resultado como array de numpy

* normalize_embeddings: normaliza los vectores a norma 1, lo que permite usar producto interno como similitud coseno (más rápido y estándar para búsqueda)

In [12]:
embeddings = modelo.encode(
    textos,
    batch_size = 128,
    show_progress_bar = True,
    convert_to_numpy = True,
    normalize_embeddings = True
)

print(f'\nShape de la matriz de embeddings: {embeddings.shape}')
print(f'Tipo de dato: {embeddings.dtype}')

Batches:   0%|          | 0/1964 [00:00<?, ?it/s]


Shape de la matriz de embeddings: (251332, 384)
Tipo de dato: float32


* Cada fila corresponde a una review, en el mismo orden que las filas del DataFrame. Es decir, la fila i de la matriz es el embedding de la review ubicada en la posición i del df.

* Cada columna es una de las 384 dimensiones del vector. En conjunto, esos 384 números son una representación numérica del significado de esa review: reviews con ideas similares tendrán vectores cercanos en este espacio, independientemente de las palabras exactas o del idioma.

Es importante destacar que estos embeddings no se almacenan dentro del DataFrame. La matriz de vectores se mantiene como un objeto separado (un array de NumPy), alineado con el df únicamente por posición. Esta separación es deliberada: las estructuras de búsqueda vectorial (como el índice FAISS que construiremos a continuación) operan directamente sobre la matriz numérica, mientras que el DataFrame conserva la información legible (texto, hotel, ciudad, etc.). El vínculo entre ambos se resuelve por el índice posicional: cuando una búsqueda devuelve la posición i, recuperamos los datos asociados con df.iloc[i].

### Indexación De Los Embeddings Con FAISS

Tenemos 251.332 vectores de 384 dimensiones. Para responder una consulta necesitamos resolver un problema clásico de nearest neighbor search: dado el vector de la consulta, encontrar los vectores más cercanos de la colección.

Para esto usamos FAISS, una librería de búsqueda por similitud sobre vectores. Como los embeddings están normalizados a norma 1, la similitud coseno equivale al producto interno, así que elegimos un índice IndexFlatIP.

IndexFlatIP no deja de ser fuerza bruta. Compara la consulta contra los 251.357 vectores sin descartar ninguno, con complejidad lineal O(n·d). No usa ninguna estructura de datos que reduzca las comparaciones. Lo único que mejora frente a un bucle ingenuo es la constante, gracias a operaciones de álgebra lineal vectorizadas que lo hacen mucho más rápido en la práctica, pero sin cambiar la complejidad asintótica.

Indexamos a nivel review. Al consultar, FAISS recupera las reviews individuales más afines a la búsqueda; como cada review pertenece a un hotel, las agrupamos por hotel y rankeamos según cuántas reviews afines tiene cada uno (y qué tan fuerte es esa afinidad). Así, un hotel sube en la recomendación cuando muchas de sus reviews se parecen a lo que el usuario busca, lo que además permite explicar la recomendación mostrando esas reviews concretas.

In [13]:
# Instalamos FAISS (versión CPU, suficiente para nuestro volumen)
%pip install faiss-cpu -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
# Importamos librería
import faiss

# FAISS espera los vectores como float32 contiguos en memoria
embeddings = np.ascontiguousarray(embeddings.astype('float32'))

# Dimensión de los vectores (384)
dim = embeddings.shape[1]

# Creamos el índice: producto interno (equivale a coseno por estar normalizados)
index = faiss.IndexFlatIP(dim)

# Agregamos todos los embeddings al índice
index.add(embeddings)

print(f'Índice construido con {index.ntotal:,} vectores de dimensión {dim}')

Índice construido con 251,332 vectores de dimensión 384


### Búsquedas De Prueba

Antes de construir el sistema de recomendación completo, validamos que la búsqueda semántica funcione correctamente sobre el índice. El objetivo de esta prueba es doble: confirmar que una consulta en lenguaje natural recupera reviews conceptualmente relacionadas (aunque no compartan las mismas palabras), y verificar que el enfoque de embeddings directos sobre el texto crudo (sin pasar por un LLM) ofrece una calidad de recuperación aceptable. Si los resultados son satisfactorios, validamos esta primera iteración; si son pobres, justificaría incorporar la etapa de extracción de ideas principales con LLM.

En esta prueba trabajamos todavía a nivel review (no agrupamos por hotel aún). Simplemente recuperamos las reviews más afines a la consulta para inspeccionar visualmente si tienen sentido.

In [15]:
def buscar_reviews(consulta, k = 5):
    """
    Recupera las k reviews más similares a una consulta en lenguaje natural.
    """
    # Vectorizamos la consulta con el MISMO modelo y la MISMA normalización que usamos para las reviews (condición necesaria para que la comparación sea válida)
    q_emb = modelo.encode(
        [consulta], # El modelo espera siempre una lista de textos, por eso los []
        convert_to_numpy = True,
        normalize_embeddings = True
    ).astype('float32')

    # FAISS devuelve los SCORES de similitud y las POSICIONES de las reviews
    scores, idxs = index.search(q_emb, k)

    # Recuperamos la info legible desde el df usando las posiciones
    print(f'Consulta: "{consulta}"\n' + '-'*60)

    posiciones = idxs[0] # Las 5 posiciones que devolvió FAISS
    puntajes = scores[0] # Los 5 scores, en el mismo orden

    for n in range(0, len(posiciones)): # n vale 0, 1, 2, 3, 4 (5 vueltas)
      i = posiciones[n] # La posición n-ésima
      score = puntajes[n] # Su score (mismo n -> van de la mano)
      rank = n + 1 # Para mostrar 1,2,3 en vez de 0,1,2

      fila = df.iloc[i] # Traemos esa fila del df una sola vez
      hotel  = fila['hotel_id_review']
      ciudad = fila['city']
      texto  = fila['texto_positivo']

      print(f'{rank}. [score = {score:.3f}] Hotel {hotel} ({ciudad})')
      print(f'   "{texto[:200]}"')

In [16]:
# Probamos con consulta 'vista al mar'
buscar_reviews("hotel tranquilo con linda vista al mar")

Consulta: "hotel tranquilo con linda vista al mar"
------------------------------------------------------------
1. [score = 0.947] Hotel 958002 (Salvador)
   "Hotel com boa localização, vista linda para o mar."
2. [score = 0.937] Hotel 307232 (Natal)
   "Hotel espaçoso, com lida vista para o mar."
3. [score = 0.928] Hotel 895491 (Rio De Janeiro)
   "Muy lindo hotel frente al mar, en una playa tranquila. Y sino la pileta con agua calentita"
4. [score = 0.928] Hotel 265488 (Cancún)
   "La vista al mar y el hotel es tranquilo ideal para descansar"
5. [score = 0.922] Hotel 359379 (Iquique)
   "Hotel buena, vista y ubicación cercana a la costanera, muy comod"


In [17]:
# Probamos con consulta 'desayuno abundante y variado'
buscar_reviews("desayuno abundante y variado")

Consulta: "desayuno abundante y variado"
------------------------------------------------------------
1. [score = 0.996] Hotel 224530 (Foz De Iguazú)
   "variado y abundante desayuno"
2. [score = 0.993] Hotel 980781 (Santiago De Chile)
   "desayuno muy variado y abundante"
3. [score = 0.986] Hotel 484858 (Florianópolis)
   "Desayuno abundante y variado"
4. [score = 0.986] Hotel 987507 (Fortaleza)
   "Desayuno abundante y variado"
5. [score = 0.986] Hotel 487165 (Rio De Janeiro)
   "Desayuno abundante y variado"


In [18]:
# Probamos con consulta 'hotel cómodo para familias con niños'
buscar_reviews("hotel cómodo para familias con niños")

Consulta: "hotel cómodo para familias con niños"
------------------------------------------------------------
1. [score = 0.925] Hotel 265301 (Cancún)
   "Hotel pensado para familias con niños"
2. [score = 0.904] Hotel 285778 (Salvador)
   "Hotel familiar con excelentes instalaciones para niños pequeños."
3. [score = 0.891] Hotel 330820 (Foz De Iguazú)
   "Pileta , y hotel familiar para ir con niños"
4. [score = 0.890] Hotel 352071 (Arraial D´ajuda)
   "La estructura hotelera perfecta para familias con niños."
5. [score = 0.889] Hotel 422842 (San Andrés)
   "Un hotel familiar y cómodo"


In [19]:
# Probamos con consulta 'hotel con buena relación precio calidad'
buscar_reviews("hotel con buena relación precio calidad")

Consulta: "hotel con buena relación precio calidad"
------------------------------------------------------------
1. [score = 0.987] Hotel 290341 (Ciudad De México)
   "Hotel de buen precio calidad"
2. [score = 0.975] Hotel 255275 (Playa Del Carmen)
   "Muy buen hotel precio calidad"
3. [score = 0.974] Hotel 1491021 (Natal)
   "Hotel de calidad y precio asequible"
4. [score = 0.963] Hotel 224586 (Puerto Iguazú)
   "La relación precio calidad que ofrece el Hotel"
5. [score = 0.962] Hotel 360064 (Foz De Iguazú)
   "Excelente hotelería y relación precio calidad"


Un aspecto destacable de los resultados es la capacidad del sistema de recuperar reviews en portugués a partir de una consulta formulada en español (y viceversa). Por ejemplo, la consulta "hotel tranquilo con linda vista al mar" recupera como primer resultado la review en portugués "Hotel com boa localização, vista linda para o mar". Esto no se debe a ninguna traducción intermedia, sino a una propiedad del modelo de embeddings elegido: al ser un modelo multilingüe, fue entrenado para proyectar frases con el mismo significado a regiones cercanas de un espacio vectorial común, independientemente del idioma. Esta característica es especialmente relevante para nuestro caso, dado que el dataset combina reviews en español y portugués en proporciones significativas: un modelo monolingüe habría ignorado cerca del 40% de la evidencia disponible en cada búsqueda. El modelo multilingüe, en cambio, permite que una consulta en cualquiera de los dos idiomas recupere las reviews más relevantes de todo el corpus.

### Agregación Por Hotel

Hasta ahora la búsqueda opera a nivel review: dada una consulta, recuperamos las reviews individuales más afines. Sin embargo, la unidad que el sistema debe recomendar es el hotel, no la review aislada. Esta sección transforma los resultados de la búsqueda semántica en un ranking de hoteles.

La estrategia consiste en recuperar un conjunto amplio de reviews afines a la consulta y luego agruparlas por hotel. Un hotel resulta más relevante cuando muchas de sus reviews se asemejan a lo que el usuario busca y, además, cuando esa semejanza es fuerte. Combinar ambos factores (cantidad de reviews afines y intensidad del match) evita dos sesgos opuestos: rankear alto a hoteles grandes solo por volumen, o a hoteles pequeños por una única review muy parecida.

Esta agregación constituye el componente de recuperación semántica del sistema de recomendación. Más adelante se integrará con el filtrado estructurado por atributos (extraídos mediante LLM) para conformar el motor de búsqueda híbrido completo.

En una primera instancia presentamos la agregación sobre el total de reviews; posteriormente la refinamos restringiendo la búsqueda al destino seleccionado por el usuario, que es el escenario de uso real del sistema.

Una vez que recuperamos las reviews más afines a la consulta, necesitamos convertir esa información en un ranking de hoteles. Para cada hotel calculamos dos señales:

* Intensidad del match (score_promedio): qué tan parecidas son sus reviews afines a lo que el usuario busca. Un promedio alto indica que las reviews realmente hablan de lo consultado.

* Consenso (reviews_afines): cuántas reviews del hotel resultaron afines. Si muchas personas mencionan lo mismo, la señal es más confiable que si lo dice una sola.

Combinamos ambas en un único puntaje:

> score_final = score_promedio × log(1 + reviews_afines)

El detalle clave está en el log. Si multiplicáramos directamente por la cantidad de reviews, un hotel con muchísimas reviews afines aplastaría siempre a uno con pocas pero muy buenas, aunque la diferencia de evidencia no sea tan grande en la práctica. El logaritmo introduce rendimientos decrecientes: pasar de 2 a 10 reviews afines aporta bastante al puntaje, pero pasar de 100 a 200 casi no mueve la aguja. Es decir, premiamos que varios usuarios coincidan, pero sin que el volumen por sí solo defina el ranking.

> **Lo que queremos es que tener "varias" reviews importe mucho, pero que tener "muchísimas" no garantice ganar por sí solo.**


In [20]:
def recomendar_hoteles(consulta, top_n = 5, k_reviews = 300):
    """
    Recomienda hoteles a partir de una consulta en lenguaje natural.

       - consulta: texto libre del usuario (ej: "hotel tranquilo con vista al mar")
       - top_n: cantidad de hoteles a devolver
       - k_reviews: cantidad de reviews afines a recuperar antes de agregar por hotel
    """

    # Vectorizamos la consulta con el mismo modelo y normalización que las reviews
    q_emb = modelo.encode(
        [consulta],
        convert_to_numpy = True,
        normalize_embeddings = True
    ).astype('float32')

    # Recuperamos las k reviews más afines a la consulta
    scores, idxs = index.search(q_emb, k_reviews)
    scores, idxs = scores[0], idxs[0]

    # Armamos un DataFrame con las reviews recuperadas y su score de similitud
    recuperadas = df.iloc[idxs].copy()
    recuperadas['score'] = scores

    # Agrupamos por hotel: para cada hotel calculamos el score promedio de sus reviews afines y cuántas reviews afines tiene
    agregado = (
        recuperadas
        .groupby('hotel_id_review')
        .agg(
            score_promedio = ('score', 'mean'),
            reviews_afines = ('score', 'size')
        )
    )

    # Score final: combinamos intensidad del match (score promedio) con consenso (cantidad de reviews), usando log para rendimientos decrecientes
    agregado['score_final'] = (
        agregado['score_promedio'] * np.log1p(agregado['reviews_afines'])
    )

    # Ordenamos y nos quedamos con el top_n
    ranking = agregado.sort_values('score_final', ascending = False).head(top_n)

    # Mostramos el resultado con reviews de ejemplo (explicabilidad)
    print(f'Consulta: "{consulta}"')
    print('=' * 70)
    for rank, (hotel_id, fila) in enumerate(ranking.iterrows(), 1):
        nombre = df.loc[df['hotel_id_review'] == hotel_id, 'name'].iloc[0]
        print(f'\n{rank}. {nombre} (Hotel {hotel_id})')
        print(f'   score_final: {fila["score_final"]:.3f} | '
              f'{int(fila["reviews_afines"])} reviews afines | '
              f'similitud media: {fila["score_promedio"]:.3f}')

        # Mostramos 2 reviews afines de ese hotel como justificación
        ejemplos = recuperadas[recuperadas['hotel_id_review'] == hotel_id]
        ejemplos = ejemplos.sort_values('score', ascending = False)['texto_positivo'].head(2)
        for ej in ejemplos:
            print(f'   - "{ej[:140]}"')

    return ranking

In [21]:
# Prueba
recomendar_hoteles("hotel tranquilo con linda vista al mar")

Consulta: "hotel tranquilo con linda vista al mar"

1. The Grand Oasis Cancún All Inclusive (Hotel 266075)
   score_final: 1.818 | 7 reviews afines | similitud media: 0.874
   - "El hotel cuenta con una hermosa vista al mar. Además posee instalaciones ideales para disfrutar todo el día"
   - "Buena ubicación en la zona hotelera, alberca grande, buenas vistas al mar."

2. Porto Seguro Eco Bahia Hotel (Hotel 866042)
   score_final: 1.518 | 5 reviews afines | similitud media: 0.847
   - "Da área Esterna do hotel, com uma vista maravilhosa da cidade e do mar"
   - "Vista del hotel y tranquilidad del lugar."

3. Hotel Nacional Rio de Janeiro OFICIAL (Hotel 895491)
   score_final: 1.417 | 4 reviews afines | similitud media: 0.880
   - "Muy lindo hotel frente al mar, en una playa tranquila. Y sino la pileta con agua calentita"
   - "Muy tranquila la playa que esta frente al hotel y la habitación es hermosa y espaciosa"

4. Wyndham Rio de Janeiro Barra (Hotel 661634)
   score_final: 1.414 | 4 

,score_promedio,reviews_afines,score_final
hotel_id_review,,,
266075,0.874218,7,1.817886
866042,0.847461,5,1.518446
895491,0.880148,4,1.416544
661634,0.878658,4,1.414146
266101,0.870462,4,1.400954


Los resultados validan el enfoque de agregación. Para la consulta "hotel tranquilo con linda vista al mar", los cinco hoteles recomendados presentan reviews que efectivamente hablan de vista al mar, tranquilidad y cercanía a la playa, recuperadas indistintamente en español y portugués. El sistema no solo entrega hoteles relevantes, sino que además acompaña cada recomendación con las reviews concretas que la justifican, cumpliendo con el requisito de explicabilidad.

El ranking obtenido muestra que la fórmula de scoring se comporta como esperábamos. El hotel ubicado en primer lugar (similitud media 0.874, 7 reviews afines) supera a hoteles con similitud media incluso más alta pero con menos reviews afines (por ejemplo, el tercer puesto con 0.880 y solo 4 reviews). Esto confirma que el sistema equilibra correctamente la intensidad del match con el consenso entre usuarios, sin que ninguno de los dos factores domine de forma aislada el resultado.

En síntesis, el componente de recuperación semántica del sistema de recomendación se encuentra funcionando de manera satisfactoria a nivel global. El siguiente paso consiste en incorporar la restricción por destino: en un escenario de uso real, el usuario primero selecciona una ciudad y luego describe lo que busca, por lo que la búsqueda debe acotarse a los hoteles de ese destino.

### Restricción Por Destino

Hasta aquí la agregación opera sobre la totalidad del dataset: una consulta recupera reviews afines de cualquier hotel, sin importar su ubicación. Sin embargo, esto no refleja el escenario de uso real del sistema. En una plataforma como Despegar o Booking, el usuario primero elige un destino y recién después describe lo que busca. No tiene sentido recomendar un hotel de Cancún a alguien que está planificando un viaje a Río de Janeiro.

Por eso incorporamos la ciudad como un filtro previo a la búsqueda semántica. La estrategia consiste en restringir el espacio de búsqueda únicamente a las reviews de los hoteles del destino seleccionado, y recién sobre ese subconjunto aplicar la recuperación semántica y la agregación por hotel descritas anteriormente.

Para implementarlo de forma eficiente aprovechamos que FAISS permite acotar la búsqueda mediante un selector de IDs (IDSelectorBatch). En lugar de buscar sobre todo el índice y descartar después los resultados de otras ciudades —lo que sería ineficiente y poco robusto si el destino tiene pocas reviews—, le indicamos a FAISS que considere exclusivamente las posiciones correspondientes a la ciudad elegida. Para esto construimos, una única vez, un diccionario que mapea cada ciudad a las posiciones de sus reviews dentro del índice; en cada consulta, recuperamos las posiciones del destino seleccionado y se las pasamos a FAISS como restricción.

El resto del proceso (cálculo del score_promedio, conteo de reviews_afines y combinación mediante score_final = score_promedio × log(1 + reviews_afines)) permanece idéntico. La única diferencia es el universo sobre el que se realiza la búsqueda: ahora acotado al destino, que es el comportamiento esperado de un sistema de recomendación de hoteles real.

In [22]:
# Construimos una sola vez el diccionario destino -> posiciones en el índice
# Cada posición coincide con la fila del df y con la fila de la matriz de embeddings: el vínculo es posicional, igual que en toda la búsqueda
from collections import defaultdict

destino_a_posiciones = defaultdict(list)
for posicion, destino in enumerate(df['destino']):
    destino_a_posiciones[destino].append(posicion)

print(f'Destinos indexados: {len(destino_a_posiciones):,}')

Destinos indexados: 67


In [23]:
def recomendar_hoteles_destino(consulta, destino, top_n = 5, k_reviews = 300):
    """
    Igual que recomendar_hoteles, pero acotando la búsqueda al destino elegido.
    El índice global NO cambia: solo le decimos a FAISS qué posiciones mirar.
    """
    # Posiciones de las reviews de ese destino
    ids_destino = np.array(destino_a_posiciones[destino], dtype = 'int64')

    # Vectorizamos la consulta con el mismo modelo y normalización
    q_emb = modelo.encode(
        [consulta],
        convert_to_numpy = True,
        normalize_embeddings = True
    ).astype('float32')

    # Selector: le indicamos a FAISS que solo considere esas posiciones
    selector = faiss.IDSelectorBatch(ids_destino)
    params = faiss.SearchParameters(sel = selector)

    # Si el destino tiene menos reviews que k_reviews, pedimos solo las que hay
    k = min(k_reviews, len(ids_destino))
    scores, idxs = index.search(q_emb, k, params = params)
    scores, idxs = scores[0], idxs[0]

    # FAISS rellena con -1 si pedimos más vecinos de los disponibles: los sacamos
    validos = idxs != -1
    idxs, scores = idxs[validos], scores[validos]

    # De acá en adelante es idéntico a recomendar_hoteles
    recuperadas = df.iloc[idxs].copy()
    recuperadas['score'] = scores

    agregado = (
        recuperadas
        .groupby('hotel_id_review')
        .agg(
            score_promedio = ('score', 'mean'),
            reviews_afines = ('score', 'size')
        )
    )
    agregado['score_final'] = (
        agregado['score_promedio'] * np.log1p(agregado['reviews_afines'])
    )
    ranking = agregado.sort_values('score_final', ascending = False).head(top_n)

    # Mostramos el resultado con reviews de ejemplo (explicabilidad)
    print(f'Destino: "{destino}"  |  Consulta: "{consulta}"')
    print('=' * 70)
    for rank, (hotel_id, fila) in enumerate(ranking.iterrows(), 1):
        nombre = df.loc[df['hotel_id_review'] == hotel_id, 'name'].iloc[0]
        print(f'\n{rank}. {nombre} (Hotel {hotel_id})')
        print(f'   score_final: {fila["score_final"]:.3f} | '
              f'{int(fila["reviews_afines"])} reviews afines | '
              f'similitud media: {fila["score_promedio"]:.3f}')
        ejemplos = recuperadas[recuperadas['hotel_id_review'] == hotel_id]
        ejemplos = ejemplos.sort_values('score', ascending = False)['texto_positivo'].head(2)
        for ej in ejemplos:
            print(f'   - "{ej[:140]}"')

    return ranking

In [24]:
# Miramos algunas claves disponibles (el string del destino tiene que ser exacto)
print(list(destino_a_posiciones)[:10])

# Probamos el escenario real: primero destino, después consulta
recomendar_hoteles_destino("hotel tranquilo con linda vista al mar",
                           destino = "Brasil - Rio De Janeiro") # Aca podemos jugar con distintos destinos

['Brasil - Gramado', 'Brasil - Rio De Janeiro', 'México - Cancún', 'Argentina - Mar Del Plata', 'Estados Unidos - Orlando', 'Brasil - Salvador', 'Brasil - Búzios', 'Brasil - Fortaleza', 'México - Acapulco', 'Uruguay - Montevideo']
Destino: "Brasil - Rio De Janeiro"  |  Consulta: "hotel tranquilo con linda vista al mar"

1. Windsor Oceanico (Hotel 835150)
   score_final: 2.643 | 26 reviews afines | similitud media: 0.802
   - "La zona donde se encuentra el hotel, segura, tranquila y cerca de la playa"
   - "las vistas de la terraza del hotel que dan al mar"

2. Wyndham Rio de Janeiro Barra (Hotel 661634)
   score_final: 2.567 | 23 reviews afines | similitud media: 0.808
   - "Hermoso hotel frente al mar."
   - "El Hotel está muy bueno con vista al mar"

3. CDesign Hotel (Hotel 876594)
   score_final: 2.445 | 20 reviews afines | similitud media: 0.803
   - "Hotel frente al mar"
   - "La ubicación del hotel frente al mar. La zona de pileta en la terraza. Las vistas desde las habitaciones 

,score_promedio,reviews_afines,score_final
hotel_id_review,,,
835150,0.801828,26,2.642695
661634,0.807822,23,2.567303
876594,0.803121,20,2.445119
231166,0.806326,16,2.284495
231814,0.808914,15,2.242786


### Migración A Vector DataBase: ChromaDB

Hasta ahora trabajamos con FAISS como motor de búsqueda vectorial. Esa primera implementación nos permitió comprender en profundidad cómo funciona la indexación y la búsqueda por similitud: construimos un índice plano con producto interno, validamos que la complejidad sigue siendo lineal en IndexFlatIP, y resolvimos el filtrado por destino mediante un IDSelectorBatch apoyado en un diccionario auxiliar.

Sin embargo, ese enfoque presenta limitaciones cuando se piensa el sistema más allá del experimento:

1. Sin persistencia: el índice vive en memoria y se pierde al reiniciar el entorno. Para volver a tener el sistema operativo hay que regenerar los embeddings y reconstruir el índice cada vez.

2. Dependencia del orden: los vectores se conectan con la metadata (hotel_id, destino, texto, etc.) únicamente por posición dentro del df y la matriz numpy. Si por cualquier motivo cambia el orden de las filas, todo el sistema se rompe silenciosamente.

3. Filtrado artesanal: para acotar la búsqueda al destino tuvimos que construir un diccionario destino → posiciones y combinarlo con IDSelectorBatch. Funciona, pero es código frágil y propio que hay que mantener.

Para resolver estos puntos migramos el sistema a ChromaDB, una vector database open-source que guarda vectores y metadata en una única estructura persistente, permite filtrado nativo por metadata directamente en la query, y elimina por completo la dependencia del orden de las filas. La lógica de scoring y agregación por hotel (el corazón de esta parte del sistema) permanece intacta, solo cambia la capa de almacenamiento y consulta de los vectores.

La base se ejecuta de forma embebida dentro del notebook (sin servidor externo) y persiste en una carpeta de Google Drive, lo que permite que sobreviva al cierre del entorno de Colab sin tener que regenerar los embeddings ni reconstruir el índice.

In [25]:
# Instalamos la librería
%pip install chromadb -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [26]:
# Importamos librería
import chromadb

# Definimos la ruta local donde ChromaDB va a persistir la base vectorial
CHROMA_PATH = Path.cwd().parent / "Data" / "VectorDB" # El notebook está en Scripts/, por eso usamos Path.cwd().parent para subir a la raíz del proyecto

# Creamos la carpeta si no existe
CHROMA_PATH.mkdir(parents = True, exist_ok = True)

# Creamos el cliente persistente de ChromaDB usando una ruta relativa al repo
cliente = chromadb.PersistentClient(path = str(CHROMA_PATH)) # Pasamos la ruta como string porque Chroma espera un path en formato texto

# Creamos o recuperamos la colección.
# Elegimos similitud coseno, coherente con la normalización de los embeddings
coleccion = cliente.get_or_create_collection(
    name = "reviews_hoteles",
    metadata = {"hnsw:space": "cosine"}
)

print(f"Colección lista: {coleccion.name}")
print(f"Ruta ChromaDB: {CHROMA_PATH}")

Colección lista: reviews_hoteles
Ruta ChromaDB: /Users/federico.hofmann/Desktop/nlp-despegar-hotel-recommendation-system/Data/VectorDB


Procedemos a cargar las 251.357 reviews en la colección. Para cada review almacenamos cuatro elementos:

1. ID único: usamos la posición de la fila en el df convertida a string. Esto asegura unicidad y simplicidad.

2. Embedding: el vector de 384 dimensiones que ya generamos con sentence-transformers.

3. Documento: el texto de la review (texto_positivo). Guardarlo dentro de Chroma nos permite recuperarlo directamente en cada query, sin tener que volver al DataFrame.

4. Metadata: los campos que vamos a usar para filtrar o mostrar al usuario (hotel_id, destino, hotel_name, city, country).

La carga se realiza en batches de 5.000 elementos. Esto se debe a que ChromaDB tiene un límite por operación de inserción; si intentamos cargar las 251k de una sola vez, falla. El batching también permite mostrar progreso visualmente. El proceso completo tarda unos minutos y se ejecuta una sola vez: la próxima vez que abras el notebook, los datos ya estarán en Chroma y solo cargás el cliente.

In [27]:
# Chequeo 1: que el df y embeddings tengan el mismo largo
print(f'Filas en df: {len(df):,}')
print(f'Vectores en embeddings: {len(embeddings):,}')
assert len(df) == len(embeddings), 'df y embeddings no están alineados!'

Filas en df: 251,332
Vectores en embeddings: 251,332


In [28]:
# Chequeo 2: que no haya valores raros en las metadatas (ChromaDB no acepta NaN)
print(df[['hotel_id_review', 'destino', 'name', 'city', 'country']].isna().sum())

hotel_id_review    0
destino            0
name               0
city               0
country            0
dtype: int64


In [29]:
# Importamos librería
from tqdm import tqdm

# Definimos el batch size (ChromaDB limita la cantidad de items por insert)
BATCH_SIZE = 5000

# Preparamos los datos en listas (ChromaDB espera listas paralelas)
ids = [str(i) for i in range(len(df))]
documentos = df['texto_positivo'].tolist()
vectores = embeddings.tolist() # numpy → lista de listas
metadatas = df[['hotel_id_review', 'destino', 'name', 'city', 'country']].rename(
    columns = {'hotel_id_review': 'hotel_id', 'name': 'hotel_name'}
).to_dict('records')

# Cargamos en batches
total = len(ids)
for i in tqdm(range(0, total, BATCH_SIZE), desc = 'Indexando reviews'):
    fin = min(i + BATCH_SIZE, total)
    coleccion.add(
        ids = ids[i:fin],
        embeddings = vectores[i:fin],
        documents = documentos[i:fin],
        metadatas = metadatas[i:fin]
    )

print(f'\nIndexación completa. Vectores en la colección: {coleccion.count():,}')

Indexando reviews: 100%|██████████| 51/51 [00:20<00:00,  2.43it/s]



Indexación completa. Vectores en la colección: 251,332


Una vez indexada la base, reescribimos la función de recomendación utilizando Chroma como motor de búsqueda. La estructura conceptual se mantiene (vectorizar la consulta, recuperar las reviews más afines del destino, agruparlas por hotel y aplicar el scoring) pero cambia significativamente la forma en la que accedemos a los datos.

El filtro por destino se realiza directamente en la query mediante where = {"destino": ...}, lo cual reemplaza tanto al diccionario auxiliar destino → posiciones como al uso de IDSelectorBatch. La traducción manual entre las posiciones devueltas por el índice y las filas del DataFrame también deja de ser necesaria: la query de Chroma retorna directamente los textos y la metadata asociados a cada review. Como consecuencia, el sistema deja de depender de que el df y la matriz de embeddings estén alineados por posición, ya que cada review se identifica de forma autónoma a través de su propio ID.

La función resultante es más concisa, más legible y robusta frente a modificaciones futuras del DataFrame.

In [30]:
def recomendar_hoteles_destino_chroma(consulta, destino, top_n = 5, k_reviews = 300):
    """
    Recomienda hoteles dentro de un destino, usando ChromaDB como motor de búsqueda.
    """
    # Vectorizamos la consulta con el mismo modelo y normalización
    q_emb = modelo.encode(
        [consulta],
        convert_to_numpy = True,
        normalize_embeddings = True
    ).tolist() # Chroma espera lista, no array

    # Consultamos Chroma con filtro nativo por destino
    resultado = coleccion.query(
        query_embeddings = q_emb,
        n_results = k_reviews,
        where = {'destino': destino}
    )

    # Chroma devuelve listas dentro de listas (una por cada query, pero acá hay 1 sola)
    distancias = resultado['distances'][0]
    documentos = resultado['documents'][0]
    metadatas  = resultado['metadatas'][0]

    # Chroma devuelve DISTANCIA coseno, convertimos a similitud: sim = 1 - dist
    similitudes = [1 - d for d in distancias]

    # Armamos un DataFrame con las reviews recuperadas
    recuperadas = pd.DataFrame({
        'hotel_id': [m['hotel_id'] for m in metadatas],
        'hotel_name': [m['hotel_name'] for m in metadatas],
        'texto': documentos,
        'score': similitudes
    })

    # Agrupamos por hotel y calculamos las dos señales
    agregado = (
        recuperadas
        .groupby(['hotel_id', 'hotel_name'])
        .agg(
            score_promedio = ('score', 'mean'),
            reviews_afines = ('score', 'size')
        )
        .reset_index()
    )

    # Score final: promedio × log(1 + cantidad)
    agregado['score_final'] = (
        agregado['score_promedio'] * np.log1p(agregado['reviews_afines'])
    )

    ranking = agregado.sort_values('score_final', ascending = False).head(top_n)

    # Mostramos el ranking con reviews de ejemplo (explicabilidad)
    print(f'Destino: "{destino}"  |  Consulta: "{consulta}"')
    print('=' * 70)
    for rank, fila in enumerate(ranking.itertuples(), 1):
        print(f'\n{rank}. {fila.hotel_name} (Hotel {fila.hotel_id})')
        print(f'   score_final: {fila.score_final:.3f} | '
              f'{int(fila.reviews_afines)} reviews afines | '
              f'similitud media: {fila.score_promedio:.3f}')

        # Reviews de ejemplo: las top 3 de ese hotel dentro de las recuperadas
        ejemplos = (recuperadas[recuperadas['hotel_id'] == fila.hotel_id]
                    .sort_values('score', ascending = False)
                    .head(3)['texto'])
        for ej in ejemplos:
            print(f'   - "{ej[:140]}"')

    return ranking

In [31]:
# Probamos la función, debería dar los mismos resultados que FAISS
recomendar_hoteles_destino_chroma("hotel tranquilo con linda vista al mar", destino = "Brasil - Rio De Janeiro")

Destino: "Brasil - Rio De Janeiro"  |  Consulta: "hotel tranquilo con linda vista al mar"

1. Windsor Oceanico (Hotel 835150)
   score_final: 2.643 | 26 reviews afines | similitud media: 0.802
   - "La zona donde se encuentra el hotel, segura, tranquila y cerca de la playa"
   - "las vistas de la terraza del hotel que dan al mar"
   - "Un hotel genial, con una ubicación ideal para quienes quieran disfrutar de la playa. La vista desde el hotel es preciosa."

2. Wyndham Rio de Janeiro Barra (Hotel 661634)
   score_final: 2.595 | 24 reviews afines | similitud media: 0.806
   - "Hermoso hotel frente al mar."
   - "El Hotel está muy bueno con vista al mar"
   - "Hotel ser perto da praia"

3. CDesign Hotel (Hotel 876594)
   score_final: 2.445 | 20 reviews afines | similitud media: 0.803
   - "Hotel frente al mar"
   - "La ubicación del hotel frente al mar. La zona de pileta en la terraza. Las vistas desde las habitaciones que dan al frente y desde la terraz"
   - "Hermoso hotel. Muy moderno.

,hotel_id,hotel_name,score_promedio,reviews_afines,score_final
50,835150,Windsor Oceanico,0.801828,26,2.642695
45,661634,Wyndham Rio de Janeiro Barra,0.806083,24,2.594681
52,876594,CDesign Hotel,0.803121,20,2.445119
21,231814,Othon Palace Copacabana Rio,0.808914,15,2.242786
53,895491,Hotel Nacional Rio de Janeiro OFICIAL,0.825778,14,2.236249
